# k-NN Baseline: Majority Voting with Embeddings

Instead of prompting an LLM, we classify by retrieving k most similar training sentences and voting on their labels.

**Question:** Is argumentativeness encoded in embedding space?

In [12]:
# Setup and imports
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent.parent))

from config.paths import GAIC_DATA_DIR
from gaic.embeddings import get_collection

# Load ChromaDB collection
collection = get_collection()
print(f"Loaded collection with {collection.count()} documents")

# Load dev data
dev_texts, dev_labels = {}, {}
with open(GAIC_DATA_DIR / "dev.jsonl") as f:
    for line in f:
        item = json.loads(line)
        dev_texts[item["id"]] = item["sentence"]
with open(GAIC_DATA_DIR / "dev_labels.jsonl") as f:
    for line in f:
        item = json.loads(line)
        dev_labels[item["id"]] = item["label"]

print(f"Loaded {len(dev_texts)} dev samples")

Loaded collection with 10200 documents
Loaded 3400 dev samples


In [13]:
def knn_predict(sentence: str, k: int, collection):
    """Predict label via k-NN majority voting."""
    results = collection.query(
        query_texts=[sentence], 
        n_results=k,
        include=["metadatas"]
    )
    labels = [m["label"] for m in results["metadatas"][0]]
    counts = Counter(labels)
    return counts.most_common(1)[0][0]


def evaluate_knn(k: int, dataset: str = None, sample_size: int = None):
    """Evaluate k-NN on dev set."""
    # Filter by dataset if specified
    if dataset:
        ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    else:
        ids = list(dev_texts.keys())
    
    # Sample if specified (balanced)
    if sample_size and sample_size < len(ids):
        arg_ids = [id_ for id_ in ids if dev_labels[id_] == "Argument"][:sample_size//2]
        noarg_ids = [id_ for id_ in ids if dev_labels[id_] == "No-Argument"][:sample_size//2]
        ids = arg_ids + noarg_ids
    
    y_true, y_pred = [], []
    for id_ in tqdm(ids, desc=f"k={k}"):
        pred = knn_predict(dev_texts[id_], k, collection)
        y_true.append(dev_labels[id_])
        y_pred.append(pred)
    
    f1 = f1_score(y_true, y_pred, average="macro")
    return f1, y_true, y_pred

## Part 1: Simple Majority Voting (all datasets)

In [14]:
# Evaluate k=50 on ALL datasets
DATASETS = ["ABSTRCT", "ACQUA", "AEC", "AFS", "ARGUMINSCI", "FINARG", "IAM", "PE", "SCIARK", "USELEC"]
K = 50
SAMPLE_SIZE = 60

all_results = []
for dataset in DATASETS:
    # Check if dataset has enough samples
    dataset_ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    if len(dataset_ids) < SAMPLE_SIZE:
        print(f"{dataset}: skipped (only {len(dataset_ids)} samples)")
        continue
    
    f1, y_true, y_pred = evaluate_knn(K, dataset=dataset, sample_size=SAMPLE_SIZE)
    all_results.append({"dataset": dataset, "k": K, "f1": f1})
    print(f"{dataset}: F1={f1:.4f}")

df_all = pd.DataFrame(all_results)
print(f"\n=== Mean F1 across datasets: {df_all['f1'].mean():.4f} ===")
df_all

k=50: 100%|██████████| 60/60 [00:11<00:00,  5.30it/s]


ABSTRCT: F1=0.8667


k=50: 100%|██████████| 60/60 [00:09<00:00,  6.06it/s]


ACQUA: F1=0.6452


k=50: 100%|██████████| 60/60 [00:09<00:00,  6.21it/s]


...

k=50: 100%|██████████| 60/60 [00:09<00:00,  6.36it/s]


SCIARK: F1=0.6475


k=50: 100%|██████████| 60/60 [00:09<00:00,  6.28it/s]

USELEC: F1=0.6832

=== Mean F1 across datasets: 0.6619 ===


,dataset,k,f1
0,ABSTRCT,50,0.866667
1,ACQUA,50,0.645170
2,AEC,50,0.490950
3,AFS,50,0.741305
4,ARGUMINSCI,50,0.691429
5,FINARG,50,0.610991
6,IAM,50,0.667735
7,PE,50,0.573943
8,SCIARK,50,0.647474
9,USELEC,50,0.683245


## Part 2: Weighted Majority Voting

Closer neighbors get more weight: `weight = 1 / (1 + distance)`

In [15]:
def knn_predict_weighted(sentence: str, k: int, collection):
    """Predict label via distance-weighted k-NN voting."""
    results = collection.query(
        query_texts=[sentence], 
        n_results=k,
        include=["metadatas", "distances"]
    )
    labels = [m["label"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    
    # Weight by inverse distance (closer = higher weight)
    weights = [1 / (1 + d) for d in distances]
    
    # Weighted vote
    label_weights = {"Argument": 0, "No-Argument": 0}
    for label, weight in zip(labels, weights):
        label_weights[label] += weight
    
    return max(label_weights, key=label_weights.get)


def evaluate_knn_weighted(k: int, dataset: str = None, sample_size: int = None):
    """Evaluate weighted k-NN on dev set."""
    if dataset:
        ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    else:
        ids = list(dev_texts.keys())
    
    if sample_size and sample_size < len(ids):
        arg_ids = [id_ for id_ in ids if dev_labels[id_] == "Argument"][:sample_size//2]
        noarg_ids = [id_ for id_ in ids if dev_labels[id_] == "No-Argument"][:sample_size//2]
        ids = arg_ids + noarg_ids
    
    y_true, y_pred = [], []
    for id_ in tqdm(ids, desc=f"weighted k={k}"):
        pred = knn_predict_weighted(dev_texts[id_], k, collection)
        y_true.append(dev_labels[id_])
        y_pred.append(pred)
    
    f1 = f1_score(y_true, y_pred, average="macro")
    return f1, y_true, y_pred

In [16]:
# Evaluate weighted k-NN (k=50) on ALL datasets
K = 50
SAMPLE_SIZE = 60

weighted_results = []
for dataset in DATASETS:
    dataset_ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    if len(dataset_ids) < SAMPLE_SIZE:
        print(f"{dataset}: skipped (only {len(dataset_ids)} samples)")
        continue
    
    f1, y_true, y_pred = evaluate_knn_weighted(K, dataset=dataset, sample_size=SAMPLE_SIZE)
    weighted_results.append({"dataset": dataset, "k": K, "f1": f1})
    print(f"{dataset}: F1={f1:.4f}")

df_weighted = pd.DataFrame(weighted_results)
print(f"\n=== Mean F1 (weighted): {df_weighted['f1'].mean():.4f} ===")
df_weighted

weighted k=50: 100%|██████████| 60/60 [00:08<00:00,  6.77it/s]


ABSTRCT: F1=0.8999


weighted k=50: 100%|██████████| 60/60 [00:09<00:00,  6.19it/s]


ACQUA: F1=0.6452


weighted k=50: 100%|██████████| 60/60 [00:09<00:00,  6.20it/s]


...

weighted k=50: 100%|██████████| 60/60 [00:10<00:00,  5.51it/s]


SCIARK: F1=0.6622


weighted k=50: 100%|██████████| 60/60 [00:09<00:00,  6.17it/s]

USELEC: F1=0.6832

=== Mean F1 (weighted): 0.6616 ===


,dataset,k,f1
0,ABSTRCT,50,0.899889
1,ACQUA,50,0.645170
2,AEC,50,0.479720
3,AFS,50,0.741305
4,ARGUMINSCI,50,0.672320
5,FINARG,50,0.610991
6,IAM,50,0.647474
7,PE,50,0.573943
8,SCIARK,50,0.662222
9,USELEC,50,0.683245


In [17]:
# Compare simple vs weighted voting
comparison = df_all.merge(df_weighted, on="dataset", suffixes=("_simple", "_weighted"))
comparison["delta"] = comparison["f1_weighted"] - comparison["f1_simple"]
comparison = comparison[["dataset", "f1_simple", "f1_weighted", "delta"]]

print("=== Simple vs Weighted k-NN (k=50) ===\n")
print(comparison.to_string(index=False))
print(f"\nMean F1 (simple):   {comparison['f1_simple'].mean():.4f}")
print(f"Mean F1 (weighted): {comparison['f1_weighted'].mean():.4f}")
print(f"Mean delta:         {comparison['delta'].mean():+.4f}")

=== Simple vs Weighted k-NN (k=50) ===

   dataset  f1_simple  f1_weighted     delta
   ABSTRCT   0.866667     0.899889  0.033222
     ACQUA   0.645170     0.645170  0.000000
       AEC   0.490950     0.479720 -0.011230
       AFS   0.741305     0.741305  0.000000
ARGUMINSCI   0.691429     0.672320 -0.019109
    FINARG   0.610991     0.610991  0.000000
       IAM   0.667735     0.647474 -0.020262
        PE   0.573943     0.573943  0.000000
    SCIARK   0.647474     0.662222  0.014749
    USELEC   0.683245     0.683245  0.000000

Mean F1 (simple):   0.6619
Mean F1 (weighted): 0.6616
Mean delta:         -0.0003


## Part 3: Per-Dataset Retrieval (in-domain only)

Retrieve only from same-dataset training samples. Tests whether in-domain retrieval outperforms cross-dataset retrieval.

In [18]:
def knn_predict_per_dataset(sentence: str, k: int, dataset: str, collection):
    """Predict label via k-NN majority voting, retrieving only from same dataset."""
    results = collection.query(
        query_texts=[sentence], 
        n_results=k,
        include=["metadatas"],
        where={"dataset": dataset}  # Filter to same dataset only
    )
    labels = [m["label"] for m in results["metadatas"][0]]
    counts = Counter(labels)
    return counts.most_common(1)[0][0]


def evaluate_knn_per_dataset(k: int, dataset: str, sample_size: int = None):
    """Evaluate k-NN on dev set, retrieving only from same dataset's train split."""
    ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    
    if sample_size and sample_size < len(ids):
        arg_ids = [id_ for id_ in ids if dev_labels[id_] == "Argument"][:sample_size//2]
        noarg_ids = [id_ for id_ in ids if dev_labels[id_] == "No-Argument"][:sample_size//2]
        ids = arg_ids + noarg_ids
    
    y_true, y_pred = [], []
    for id_ in tqdm(ids, desc=f"per-dataset k={k}"):
        pred = knn_predict_per_dataset(dev_texts[id_], k, dataset, collection)
        y_true.append(dev_labels[id_])
        y_pred.append(pred)
    
    f1 = f1_score(y_true, y_pred, average="macro")
    return f1, y_true, y_pred

In [19]:
# Evaluate per-dataset k-NN (k=50) on ALL datasets
K = 50
SAMPLE_SIZE = 60

per_dataset_results = []
for dataset in DATASETS:
    dataset_ids = [id_ for id_ in dev_texts if id_.startswith(dataset)]
    if len(dataset_ids) < SAMPLE_SIZE:
        print(f"{dataset}: skipped (only {len(dataset_ids)} samples)")
        continue
    
    f1, y_true, y_pred = evaluate_knn_per_dataset(K, dataset=dataset, sample_size=SAMPLE_SIZE)
    per_dataset_results.append({"dataset": dataset, "k": K, "f1": f1})
    print(f"{dataset}: F1={f1:.4f}")

df_per_dataset = pd.DataFrame(per_dataset_results)
print(f"\n=== Mean F1 (per-dataset retrieval): {df_per_dataset['f1'].mean():.4f} ===")
df_per_dataset

per-dataset k=50: 100%|██████████| 60/60 [00:10<00:00,  5.88it/s]


ABSTRCT: F1=0.8667


per-dataset k=50: 100%|██████████| 60/60 [00:10<00:00,  5.82it/s]


ACQUA: F1=0.6606


per-dataset k=50: 100%|██████████| 60/60 [00:11<00:00,  5.37it/s]


...

per-dataset k=50: 100%|██████████| 60/60 [00:14<00:00,  4.14it/s]


SCIARK: F1=0.7129


per-dataset k=50: 100%|██████████| 60/60 [00:11<00:00,  5.13it/s]

USELEC: F1=0.6946

=== Mean F1 (per-dataset retrieval): 0.6767 ===


,dataset,k,f1
0,ABSTRCT,50,0.866667
1,ACQUA,50,0.660633
2,AEC,50,0.597785
3,AFS,50,0.760000
4,ARGUMINSCI,50,0.691429
5,FINARG,50,0.610991
6,IAM,50,0.597785
7,PE,50,0.573943
8,SCIARK,50,0.712919
9,USELEC,50,0.694570


In [20]:
# Compare all three approaches: cross-dataset vs weighted vs per-dataset
comparison_all = df_all.merge(df_weighted, on="dataset", suffixes=("_cross", "_weighted"))
comparison_all = comparison_all.merge(df_per_dataset, on="dataset")
comparison_all = comparison_all.rename(columns={"f1": "f1_per_dataset"})
comparison_all["delta_per_vs_cross"] = comparison_all["f1_per_dataset"] - comparison_all["f1_cross"]
comparison_all = comparison_all[["dataset", "f1_cross", "f1_weighted", "f1_per_dataset", "delta_per_vs_cross"]]

print("=== k-NN Comparison (k=50): Cross-Dataset vs Per-Dataset Retrieval ===\n")
print(comparison_all.to_string(index=False))
print(f"\nMean F1 (cross-dataset): {comparison_all['f1_cross'].mean():.4f}")
print(f"Mean F1 (weighted):      {comparison_all['f1_weighted'].mean():.4f}")
print(f"Mean F1 (per-dataset):   {comparison_all['f1_per_dataset'].mean():.4f}")
print(f"Mean delta (per-cross):  {comparison_all['delta_per_vs_cross'].mean():+.4f}")

=== k-NN Comparison (k=50): Cross-Dataset vs Per-Dataset Retrieval ===

   dataset  f1_cross  f1_weighted  f1_per_dataset  delta_per_vs_cross
   ABSTRCT  0.866667     0.899889        0.866667            0.000000
     ACQUA  0.645170     0.645170        0.660633            0.015463
       AEC  0.490950     0.479720        0.597785            0.106835
       AFS  0.741305     0.741305        0.760000            0.018695
ARGUMINSCI  0.691429     0.672320        0.691429            0.000000
    FINARG  0.610991     0.610991        0.610991            0.000000
       IAM  0.667735     0.647474        0.597785           -0.069950
        PE  0.573943     0.573943        0.573943            0.000000
    SCIARK  0.647474     0.662222        0.712919            0.065445
    USELEC  0.683245     0.683245        0.694570            0.011325

Mean F1 (cross-dataset): 0.6619
Mean F1 (weighted):      0.6616
Mean F1 (per-dataset):   0.6767
Mean delta (per-cross):  +0.0148
